# 01 — Ingestion ESCO

On va filtrer les professions de la distribution **française officielle d'ESCO** (`Datasets/ESCO/*_fr.csv`),  du périmètre informatique via leur code ISCO, récupère leurs compétences (essentielles/optionnelles), et écrit le tout dans la couche canonique CSV.

In [1]:
import pandas as pd
import jobkb_common as C
from collections import Counter

SOURCE = "ESCO"
ESCO_VERSION = "v1.2 (fr)"

# pour les libellés multiples d'ESCO, on les sépare par des retours ligne dans une cellule
def split_multi(cell):
    return [x.strip() for x in (cell or "").split("\n") if x.strip()]

## 1.1. Chargement des fichiers ESCO français

In [3]:
import os 

occ_df    = C.read_csv_smart(os.path.join(C.ESCO_DIR, "occupations_fr.csv"))
skl_df    = C.read_csv_smart(os.path.join(C.ESCO_DIR, "skills_fr.csv"))
rel_df    = C.read_csv_smart(os.path.join(C.ESCO_DIR, "occupationSkillRelations_fr.csv"))
transv_df = C.read_csv_smart(os.path.join(C.ESCO_DIR, "transversalSkillsCollection_fr.csv"))

print("occupations_fr :", occ_df.shape)
print("skills_fr      :", skl_df.shape)
print("relations_fr   :", rel_df.shape)
print("transversal    :", transv_df.shape, "(collection soft curatée d'ESCO)")

# URIs de la collection transversale, pour filtrer les compétences transversales dans la table des relations
transv_uris = set(transv_df["conceptUri"])

occupations_fr : (3043, 15)
skills_fr      : (13960, 13)
relations_fr   : (126051, 6)
transversal    : (95, 10) (collection soft curatée d'ESCO)


## 1.2. Filtrage des professions au périmètre informatique (via code ISCO)

In [4]:
occ_df["iscoGroup"] = occ_df["iscoGroup"].astype(str).str.strip()
in_scope = occ_df[occ_df["iscoGroup"].isin(C.ISCO_UNIT_GROUPS_IN_SCOPE)].copy()

print(f"Professions ESCO dans le périmètre : {len(in_scope)}  (sur {len(occ_df)} au total)")
print("\nRépartition par groupe ISCO :")
for code_, n in sorted(Counter(in_scope["iscoGroup"]).items()):
    print(f"   {code_}  {C.ISCO_UNIT_GROUPS_IN_SCOPE[code_][:45]:45s}  {n}")

print("\nExemples (libellés français) :")
for _, r in in_scope.head(6).iterrows():
    print("   -", r["preferredLabel"])

Professions ESCO dans le périmètre : 83  (sur 3043 au total)

Répartition par groupe ISCO :
   2511  Analystes de systemes                          21
   2512  Concepteurs de logiciels                       10
   2513  Concepteurs de sites internet et d'outils mul  5
   2514  Programmeurs d'applications                    7
   2519  Concepteurs et analystes de logiciels, et con  12
   2521  Concepteurs et administrateurs de bases de do  5
   2522  Administrateurs de systemes                    3
   2523  Ingenieurs et specialistes des reseaux inform  3
   2529  Ingenieurs et specialistes des bases de donne  9
   3511  Techniciens TIC, maintenance                   1
   3512  Techniciens TIC, assistance aux utilisateurs   4
   3513  Techniciens, reseaux et systemes informatique  2
   3514  Techniciens de l'internet                      1

Exemples (libellés français) :
   - administrateur sécurité informatique/administratrice sécurité informatique
   - analyste logiciel
   - ingénieur i

## 1.3. Construction des professions canoniques + labels

In [5]:
occ_id = {}          # conceptUri -> entity_id
occ_rows, label_rows = [], []

for _, o in in_scope.iterrows():
    sid = C.uri_tail(o["conceptUri"])
    eid = C.mint_id("OCC_", SOURCE, sid)
    occ_id[o["conceptUri"]] = eid
    alts   = split_multi(o["altLabels"])
    hidden = split_multi(o["hiddenLabels"])
    occ_rows.append({
        "entity_id": eid, "source": SOURCE, "source_id": sid,
        "isco_code": o["iscoGroup"],
        "pref_label_fr": o["preferredLabel"], "pref_label_en": "",
        "alt_labels_fr": " | ".join(alts), "alt_labels_en": "",
        "description_fr": (o["description"] or "").replace("\n", " "), "description_en": "",
        "occupation_type": o["conceptType"], "label_language_status": "fr_native",
    })
    label_rows += C.make_label_rows(eid, "occupation", SOURCE,
        preferred={"fr": [o["preferredLabel"]]},
        alts={"fr": alts}, hidden={"fr": hidden})

print(f"{len(occ_rows)} professions, {len(label_rows)} labels (professions) construits")

83 professions, 715 labels (professions) construits


## 1.4. Compétences liées aux professions du périmètre (informatique) + classification hard/soft

La classification hard/soft s'appuie sur la *structure explicite* d'ESCO :
- membre de la **collection transversale** curatée → **soft**
- pilier **knowledge** → **hard** (une connaissance est toujours *hard*)
- sinon **skill/competence** → **hard** par défaut

In [6]:
rel_scope  = rel_df[rel_df["occupationUri"].isin(occ_id)]
skill_uris = set(rel_scope["skillUri"])
skl_by_uri = {r["conceptUri"]: r for _, r in skl_df.iterrows()}

def classify_hard_soft(uri, skilltype):
    if uri in transv_uris:
        return "soft", "esco_transversal_collection"
    if skilltype == "knowledge":
        return "hard", "esco_knowledge_pillar"
    return "hard", "esco_skillcompetence_default"

skl_id = {}
skl_rows = []
for uri in skill_uris:
    s = skl_by_uri.get(uri)
    if s is None:
        continue
    sid = C.uri_tail(uri)
    eid = C.mint_id("SKL_", SOURCE, sid)
    skl_id[uri] = eid
    alts   = split_multi(s["altLabels"])
    hidden = split_multi(s["hiddenLabels"])
    hs, method = classify_hard_soft(uri, s["skillType"])
    skl_rows.append({
        "entity_id": eid, "source": SOURCE, "source_id": sid,
        "pref_label_fr": s["preferredLabel"], "pref_label_en": "",
        "alt_labels_fr": " | ".join(alts), "alt_labels_en": "",
        "description_fr": (s["description"] or "").replace("\n", " "), "description_en": "",
        "esco_skill_type": s["skillType"], "esco_reuse_level": s["reuseLevel"],
        "hard_soft_provisional": hs, "hard_soft_method": method,
    })
    label_rows += C.make_label_rows(eid, "skill", SOURCE,
        preferred={"fr": [s["preferredLabel"]]},
        alts={"fr": alts}, hidden={"fr": hidden})

print(f"{len(skl_rows)} compétences liées au périmètre")
print("hard/soft :", dict(Counter(r["hard_soft_provisional"] for r in skl_rows)))

925 compétences liées au périmètre
hard/soft : {'hard': 924, 'soft': 1}


## 1.5. Relations profession → compétence (essentielle / optionnelle)

In [7]:
rel_rows = [{
    "occupation_entity_id": occ_id[r["occupationUri"]],
    "skill_entity_id": skl_id[r["skillUri"]],
    "relation_type": r["relationType"],   # 'essential' | 'optional'
    "source": SOURCE,
} for _, r in rel_scope.iterrows() if r["skillUri"] in skl_id]

print(f"{len(rel_rows)} relations profession→compétence")
print("types :", dict(Counter(r["relation_type"] for r in rel_rows)))

5322 relations profession→compétence
types : {'essential': 1960, 'optional': 3362}


## 1.6. Écriture au format canonique

In [8]:
C.replace_source_rows(C.OCCUPATIONS_CSV,   C.OCCUPATION_FIELDS, SOURCE, occ_rows)
C.replace_source_rows(C.SKILLS_CSV,        C.SKILL_FIELDS,      SOURCE, skl_rows)
C.replace_source_rows(C.LABELS_CSV,        C.LABEL_FIELDS,      SOURCE, label_rows)
C.replace_source_rows(C.OCC_SKILL_REL_CSV, C.REL_FIELDS,        SOURCE, rel_rows)
C.log_provenance(SOURCE, [{
    "entity_id": "ALL_ESCO", "source": SOURCE, "source_version": ESCO_VERSION,
    "retrieved_at": C.now_iso(), "retrieval_method": "official_fr_csv",
    "notes": f"{len(occ_rows)} occ, {len(skl_rows)} skills, {len(label_rows)} labels, {len(rel_rows)} rel",
}])
print("Écrit dans", C.CANONICAL_DIR)
for p in [C.OCCUPATIONS_CSV, C.SKILLS_CSV, C.LABELS_CSV, C.OCC_SKILL_REL_CSV]:
    print("   -", os.path.basename(p))

Écrit dans d:\JobKB-final\canonical
   - occupations.csv
   - skills.csv
   - labels.csv
   - occupation_skill_relations.csv


## 1.7. Vérifications

In [9]:
import csv as _csv
_csv.field_size_limit(10_000_000)

def load(path):
    with open(path, encoding="utf-8") as f:
        return list(_csv.DictReader(f))

occs   = [r for r in load(C.OCCUPATIONS_CSV) if r["source"] == SOURCE]
skills = [r for r in load(C.SKILLS_CSV)      if r["source"] == SOURCE]
labels = [r for r in load(C.LABELS_CSV)      if r["source"] == SOURCE]
rels   = [r for r in load(C.OCC_SKILL_REL_CSV) if r["source"] == SOURCE]

occ_ids = {o["entity_id"] for o in occs}
skl_ids = {s["entity_id"] for s in skills}

# intégrité référentielle
dangling = [r for r in rels if r["occupation_entity_id"] not in occ_ids or r["skill_entity_id"] not in skl_ids]
assert not dangling, f"{len(dangling)} relations pendantes !"
lab_dangling = [l for l in labels if l["entity_id"] not in (occ_ids | skl_ids)]
assert not lab_dangling, f"{len(lab_dangling)} labels pendants !"
print("Intégrité référentielle : OK (aucune relation/label pendant)")

# richesse des alias
print("\nLabels par type :", dict(Counter(l["label_type"] for l in labels)))
print("Labels par langue :", dict(Counter(l["language"] for l in labels)))
n_alias = sum(1 for l in labels if l["label_type"] in ("alt", "hidden"))
print(f"Alias/synonymes : {n_alias}  (~{n_alias/max(len(occ_ids|skl_ids),1):.1f} par entité)")

Intégrité référentielle : OK (aucune relation/label pendant)

Labels par type : {'preferred': 1008, 'alt': 2066, 'hidden': 457}
Labels par langue : {'fr': 3531}
Alias/synonymes : 2523  (~2.5 par entité)


In [11]:
# Exemple
target = next((o for o in occs if o["isco_code"] == "2512"
               and "analyste" in o["pref_label_fr"].lower()), occs[0])
aliases = [l["label_text"] for l in labels if l["entity_id"] == target["entity_id"]]
print(f"« {target['pref_label_fr']} » (ISCO {target['isco_code']}) — libellés français :")
for a in aliases:
    print("   -", a)

« analyste logiciel » (ISCO 2512) — libellés français :
   - analyste logiciel
   - analyste informatique
   - analyste de logiciels
   - analyste d'applications
   - analyste software


----

# 02 — Ingestion ISCO (français)

ISCO-08 est le squelette hiérarchique sous lequel se rangent les professions ESCO.

La hiérarchie ISCO se déduit de l'emboîtement des codes (2512 ⊂ 251 ⊂ 25 ⊂ 2), pas d'une reconstruction hasardeuse.

In [12]:
SOURCE = "ISCO"
ISCO_VERSION = "ISCO-08 (nomenclature FR 4 niveaux)"

## 2.1. Chargement de la nomenclature ISCO

In [13]:
isco_df = C.read_csv_smart(os.path.join(C.ISCO_DIR, "nomenclature_4N_emboites_ISCO.csv"))
isco_df["Niveau"] = isco_df["Niveau"].astype(str).str.strip()
isco_df["Code ISCO"] = isco_df["Code ISCO"].astype(str).str.strip()
# code réel = n premiers caractères, avec n = niveau
isco_df["isco_real"] = isco_df.apply(lambda r: r["Code ISCO"][:int(r["Niveau"])], axis=1)

label_by_code = {r["isco_real"]: r["Libellé"] for _, r in isco_df.iterrows()}
level_by_code = {r["isco_real"]: int(r["Niveau"]) for _, r in isco_df.iterrows()}
print("Nomenclature ISCO :", isco_df.shape[0], "lignes, 4 niveaux")
print("Exemple de chaîne : 2512 ->", " <- ".join(
    f"{c}" for c in ["2512","251","25","2"]))

Nomenclature ISCO : 620 lignes, 4 niveaux
Exemple de chaîne : 2512 -> 2512 <- 251 <- 25 <- 2


## 2.2. Charger les professions ESCO crées

In [14]:
import csv

def load_source(path, source):
    if not os.path.isfile(path):
        return []
    with open(path, encoding="utf-8") as f:
        return [r for r in csv.DictReader(f) if r.get("source") == source]

esco_occs = load_source(C.OCCUPATIONS_CSV, "ESCO")
assert esco_occs, "Aucune profession ESCO trouvée."
print(f"Professions ESCO à rattacher : {len(esco_occs)}")

def ancestors(code):
    """Chaîne d'ancêtres d'un code ISCO, du plus fin au grand groupe."""
    out, c = [], code
    while c:
        out.append(c)
        c = c[:-1] if len(c) > 1 else None
    return out

needed_codes = set()
for o in esco_occs:
    if o["isco_code"]:
        needed_codes.update(ancestors(o["isco_code"]))
print(f"Nœuds de groupes ISCO nécessaires (tous niveaux) : {len(needed_codes)}")
missing = [c for c in needed_codes if c not in label_by_code]
assert not missing, f"Codes ISCO sans libellé FR : {missing}"
print("Tous les groupes nécessaires ont un libellé français.")

Professions ESCO à rattacher : 83
Nœuds de groupes ISCO nécessaires (tous niveaux) : 20
Tous les groupes nécessaires ont un libellé français.


## 2.3. Créer les nœuds « groupe ISCO »

Les groupes ISCO sont modélisés comme des professions de type `isco_group` (ce sont des regroupements de professions). Ils reçoivent un `entity_id` déterministe et un libellé.

In [15]:
isco_occ_rows, label_rows = [], []
isco_entity_id = {}   # code ISCO -> entity_id

for code_ in sorted(needed_codes):
    eid = C.mint_id("OCC_", SOURCE, code_)
    isco_entity_id[code_] = eid
    lib = label_by_code[code_]
    isco_occ_rows.append({
        "entity_id": eid, "source": SOURCE, "source_id": code_,
        "isco_code": code_,
        "pref_label_fr": lib, "pref_label_en": "",
        "alt_labels_fr": "", "alt_labels_en": "",
        "description_fr": f"Groupe ISCO-08 niveau {level_by_code[code_]}", "description_en": "",
        "occupation_type": "isco_group", "label_language_status": "fr_native",
    })
    label_rows += C.make_label_rows(eid, "occupation", SOURCE, preferred={"fr": [lib]})

print(f"{len(isco_occ_rows)} nœuds de groupes ISCO créés")
for r in isco_occ_rows[:6]:
    print(f"   {r['isco_code']:5s} {r['pref_label_fr'][:55]}")

20 nœuds de groupes ISCO créés
   2     Professions intellectuelles et scientifiques
   25    Ingénieurs et professionnels des technologies de l’info
   251   Analystes et concepteurs de logiciels et d'outils multi
   2511  Analystes de systèmes
   2512  Concepteurs de logiciels
   2513  Concepteurs de sites internet et d'outils multimédias


## 2.4. Créer les arêtes hiérarchiques

Deux types d'arêtes :
- **groupe ISCO → groupe ISCO parent** (2512 → 251 → 25 → 2)
- **profession ESCO → son groupe de base ISCO** (rattachement des feuilles)

In [16]:
hier_rows = []

# arêtes internes à l'arbre ISCO (parent = code sans le dernier chiffre)
for code_ in needed_codes:
    parent = code_[:-1] if len(code_) > 1 else None
    if parent and parent in isco_entity_id:
        hier_rows.append({
            "parent_entity_id": isco_entity_id[parent],
            "child_entity_id": isco_entity_id[code_],
            "entity_kind": "occupation", "relation_type": "broader_than", "source": SOURCE,
        })

# rattacher chaque profession ESCO à son groupe de base ISCO
attached = 0
for o in esco_occs:
    grp = o["isco_code"]
    if grp in isco_entity_id:
        hier_rows.append({
            "parent_entity_id": isco_entity_id[grp],
            "child_entity_id": o["entity_id"],
            "entity_kind": "occupation", "relation_type": "broader_than", "source": SOURCE,
        })
        attached += 1

print(f"{len(hier_rows)} arêtes hiérarchiques "
      f"({len(hier_rows)-attached} internes ISCO + {attached} rattachements ESCO)")

101 arêtes hiérarchiques (18 internes ISCO + 83 rattachements ESCO)


## 2.5. Écriture au format canonique

In [17]:
# les nœuds groupes ISCO s'ajoutent à occupations.csv (source=ISCO) ;
# ESCO reste intact (remplacement PAR SOURCE)

C.replace_source_rows(C.OCCUPATIONS_CSV, C.OCCUPATION_FIELDS, SOURCE, isco_occ_rows)
C.replace_source_rows(C.LABELS_CSV,      C.LABEL_FIELDS,      SOURCE, label_rows)
C.replace_source_rows(C.HIERARCHY_CSV,   C.HIERARCHY_FIELDS,  SOURCE, hier_rows)
C.log_provenance(SOURCE, [{
    "entity_id": "ALL_ISCO", "source": SOURCE, "source_version": ISCO_VERSION,
    "retrieved_at": C.now_iso(), "retrieval_method": "nomenclature_4N_fr_csv",
    "notes": f"{len(isco_occ_rows)} groupes ISCO, {len(hier_rows)} arêtes hiérarchiques",
}])
print("Écrit : occupations.csv (+ISCO), labels.csv (+ISCO), hierarchy.csv (ISCO)")

Écrit : occupations.csv (+ISCO), labels.csv (+ISCO), hierarchy.csv (ISCO)


## 2.6. Vérifications

On vérifie qu'aucune arête ne pendouille et on affiche une branche complète de l'arbre

In [ ]:
def load_all(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

all_occs = load_all(C.OCCUPATIONS_CSV)
all_hier = [h for h in load_all(C.HIERARCHY_CSV) if h["source"] == SOURCE]
id2occ = {o["entity_id"]: o for o in all_occs}

# intégrité : toutes les extrémités existent
dangling = [h for h in all_hier
            if h["parent_entity_id"] not in id2occ or h["child_entity_id"] not in id2occ]
assert not dangling, f"{len(dangling)} arêtes hiérarchiques pendantes !"
print("Intégrité hiérarchique : OK")
print("Total professions (ESCO + groupes ISCO) :", len(all_occs))
print("Arêtes hiérarchiques ISCO :", len(all_hier))

Intégrité hiérarchique : OK
Total professions (ESCO + groupes ISCO) : 103
Arêtes hiérarchiques ISCO : 101


In [20]:
# Afficher une branche : d'une profession ESCO jusqu'au grand groupe ISCO
parent_of = {h["child_entity_id"]: h["parent_entity_id"] for h in all_hier}

leaf = next(o for o in all_occs if o["source"] == "ESCO" and o["isco_code"] == "2512")
chain, cur = [], leaf["entity_id"]
while cur:
    o = id2occ[cur]
    tag = "ISCO" if o["occupation_type"] == "isco_group" else "profession ESCO"
    chain.append(f"[{o['isco_code'] or '—':4s}] {o['pref_label_fr'][:50]}  ({tag})")
    cur = parent_of.get(cur)

print("Branche hiérarchique (feuille → racine) :\n")
for i, node in enumerate(chain):
    print("   " * i + "└─ " + node)

Branche hiérarchique (feuille → racine) :

└─ [2512] analyste logiciel  (profession ESCO)
   └─ [2512] Concepteurs de logiciels  (ISCO)
      └─ [251 ] Analystes et concepteurs de logiciels et d'outils   (ISCO)
         └─ [25  ] Ingénieurs et professionnels des technologies de l  (ISCO)
            └─ [2   ] Professions intellectuelles et scientifiques  (ISCO)
